In [1]:
import pandas as pd
import requests as rq
import sqlite3 as sq

In [2]:
rate_URl = "https://api.frankfurter.dev/v2/rates?base=USD&from=2024-01-01"
currency_URL = "https://api.frankfurter.dev/v2/currencies?scope=all"

In [3]:
# Load the exchange rate data from the API
json_file = rq.get(url=rate_URl)
fact_data = pd.DataFrame(json_file.json())
fact_data


,date,base,quote,rate
0,2023-12-29,USD,ANG,1.79000
1,2023-12-29,USD,AOA,839.06000
2,2023-12-29,USD,ARS,808.64000
3,2023-12-29,USD,AWG,1.79890
4,2023-12-29,USD,BBD,2.03680
...,...,...,...,...
134578,2026-08-17,USD,XPT,0.00057
134579,2026-08-17,USD,YER,236.99000
134580,2026-08-17,USD,ZAR,16.15290
134581,2026-08-17,USD,ZMW,18.82100


In [4]:
# Load the currency dimension data from the API
currency_dim = rq.get(url=currency_URL)
currency_dim = pd.DataFrame(currency_dim.json())
currency_dim

,iso_code,iso_numeric,name,symbol,start_date,end_date
0,AED,784,United Arab Emirates Dirham,د.إ,1996-04-11,2026-08-17
1,AFN,971,Afghan Afghani,؋,1999-01-04,2026-08-17
2,ALL,008,Albanian Lek,L,1998-07-07,2026-08-17
3,AMD,051,Armenian Dram,֏,1994-03-31,2026-08-17
4,ANG,532,Netherlands Antillean Gulden,ƒ,1971-12-18,2026-08-17
...,...,...,...,...,...,...
196,ZWD,716,Zimbabwean Dollar,$,1999-01-04,2013-10-30
197,ZWG,924,Zimbabwe Gold,ZiG,2024-09-02,2026-08-17
198,ZWL,932,Zimbabwean Dollar,$,2010-06-02,2024-08-30
199,ZWN,942,Zimbabwean Dollar,$,2006-09-01,2006-10-25


In [5]:
currency = []
splitted = currency_dim["name"].str.split(" ",n=3)
for i in splitted:
    currency.append(i[-1])
currency_dim["Currency"] = currency

In [6]:
currency_dim

,iso_code,iso_numeric,name,symbol,start_date,end_date,Currency
0,AED,784,United Arab Emirates Dirham,د.إ,1996-04-11,2026-08-17,Dirham
1,AFN,971,Afghan Afghani,؋,1999-01-04,2026-08-17,Afghani
2,ALL,008,Albanian Lek,L,1998-07-07,2026-08-17,Lek
3,AMD,051,Armenian Dram,֏,1994-03-31,2026-08-17,Dram
4,ANG,532,Netherlands Antillean Gulden,ƒ,1971-12-18,2026-08-17,Gulden
...,...,...,...,...,...,...,...
196,ZWD,716,Zimbabwean Dollar,$,1999-01-04,2013-10-30,Dollar
197,ZWG,924,Zimbabwe Gold,ZiG,2024-09-02,2026-08-17,Gold
198,ZWL,932,Zimbabwean Dollar,$,2010-06-02,2024-08-30,Dollar
199,ZWN,942,Zimbabwean Dollar,$,2006-09-01,2006-10-25,Dollar


In [7]:
country = []
splitted2 = currency_dim["name"].str.split(" ",n=3)
for i in splitted2:
    if len(i) == 4:
        country.append(i[0] + " " + i[1] + " " + i[2])
    elif len(i) == 3:
        country.append(i[0] + " " + i[1])
    else:
        country.append(i[0])
currency_dim["Country"] = country

In [8]:
currency_dim.drop(columns="name",inplace=True)
currency_dim = currency_dim[["start_date","end_date","Country","Currency","symbol","iso_code","iso_numeric"]]
currency_dim

,start_date,end_date,Country,Currency,symbol,iso_code,iso_numeric
0,1996-04-11,2026-08-17,United Arab Emirates,Dirham,د.إ,AED,784
1,1999-01-04,2026-08-17,Afghan,Afghani,؋,AFN,971
2,1998-07-07,2026-08-17,Albanian,Lek,L,ALL,008
3,1994-03-31,2026-08-17,Armenian,Dram,֏,AMD,051
4,1971-12-18,2026-08-17,Netherlands Antillean,Gulden,ƒ,ANG,532
...,...,...,...,...,...,...,...
196,1999-01-04,2013-10-30,Zimbabwean,Dollar,$,ZWD,716
197,2024-09-02,2026-08-17,Zimbabwe,Gold,ZiG,ZWG,924
198,2010-06-02,2024-08-30,Zimbabwean,Dollar,$,ZWL,932
199,2006-09-01,2006-10-25,Zimbabwean,Dollar,$,ZWN,942


In [9]:
conn = sq.connect("exchange_rates.db")
fact_data.to_sql("Exchange_Rates",con=conn, if_exists="replace", index=False)
currency_dim.to_sql("currency_dim",con=conn, if_exists="replace", index=False)
conn.close()